In [17]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time


headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}


all_data = []


# Offsets for pages (3 pages = ~600 movies)
offsets = [0, 200, 400]


for off in offsets:

    url = f"https://www.boxofficemojo.com/chart/ww_top_lifetime_gross/?offset={off}"

    print("Scraping:", url)

    res = requests.get(url, headers=headers)
    soup = BeautifulSoup(res.text, "html.parser")
    soup.raise_for_status()

    rows = soup.find_all("tr")


    for movie in rows[1:]:

        try:

            rank = movie.find(
                "td",
                class_="a-text-right mojo-header-column mojo-truncate mojo-field-type-rank"
            ).text.strip()

            name = movie.find(
                "a",
                class_="a-link-normal"
            ).text.strip()

            money = movie.find_all(
                "td",
                class_="a-text-right mojo-field-type-money"
            )

            worldwide = money[0].text.strip()
            domestic = money[1].text.strip()
            foreign = money[2].text.strip()

            percent = movie.find_all(
                "td",
                class_="a-text-right mojo-field-type-percent"
            )

            dom_pct = percent[0].text.strip()
            for_pct = percent[1].text.strip()

            year = movie.find(
                "td",
                class_="a-text-left mojo-field-type-year"
            ).text.strip()


            all_data.append([
                rank, name, worldwide,
                domestic, foreign,
                dom_pct, for_pct, year
            ])


        except:
            continue


    time.sleep(2)   


df = pd.DataFrame(all_data, columns=[
    "Rank", "Name", "Worldwide Gross",
    "Domestic Gross", "Foreign Gross",
    "Domestic %", "Foreign %", "Year"
])


df = df.drop_duplicates(subset=["Name"])



df

Scraping: https://www.boxofficemojo.com/chart/ww_top_lifetime_gross/?offset=0
Scraping: https://www.boxofficemojo.com/chart/ww_top_lifetime_gross/?offset=200
Scraping: https://www.boxofficemojo.com/chart/ww_top_lifetime_gross/?offset=400


,Rank,Name,Worldwide Gross,Domestic Gross,Foreign Gross,Domestic %,Foreign %,Year
0,1,Avatar,"$2,923,710,708","$785,221,649","$2,138,489,059",26.9%,73.1%,2009
1,2,Avengers: Endgame,"$2,799,439,100","$858,373,000","$1,941,066,100",30.7%,69.3%,2019
2,3,Avatar: The Way of Water,"$2,334,484,620","$688,459,501","$1,646,025,119",29.5%,70.5%,2022
3,4,Titanic,"$2,264,812,968","$674,354,882","$1,590,458,086",29.8%,70.2%,1997
4,5,Ne Zha 2,"$2,259,822,417","$23,308,176","$2,236,514,241",1%,99%,2025
...,...,...,...,...,...,...,...,...
595,596,Resident Evil: Afterlife,"$300,228,084","$60,128,566","$240,099,518",20%,80%,2010
596,597,Migration,"$300,187,799","$127,630,880","$172,556,919",42.5%,57.5%,2023
597,598,Van Helsing,"$300,157,638","$120,177,084","$179,980,554",40%,60%,2004
598,599,Stuart Little,"$300,135,367","$140,035,367","$160,100,000",46.7%,53.3%,1999


In [26]:
df.to_csv("boxoffice_593_movies.csv", index=False)